# Práctica 01 · AgentOps y LLMOps con MLflow

**Versión para completar.** Instrumenta, etiqueta, filtra y evalúa un asistente didáctico con tres *scorers* de código y fallos trazados. El modo obligatorio es determinista, por lo que funciona en Databricks Free Edition sin llamar a un LLM.

> No envíes datos de pacientes, secretos ni información personal a trazas o endpoints. El agente no ofrece consejo clínico.

## 0. Dependencias y configuración

Actualiza MLflow si fuese necesario. Para la práctica deja `USE_LLM = False`. Sólo si el docente lo indica y tu cuenta tiene capacidad, podrás instalar `databricks-openai` y configurar un endpoint autorizado.

In [0]:
pip install -q --upgrade "mlflow[databricks]" "pandas<3"

In [0]:
dbutils.library.restartPython()  # reinicia Python para usar la versión nueva de mlflow

In [0]:
import json
from typing import Any

import mlflow
from mlflow.genai.scorers import scorer

USE_LLM = False
AGENT_VERSION = "v1-deterministic"
MODEL_ENDPOINT = "REEMPLAZA_CON_UN_ENDPOINT_AUTORIZADO"
mlflow.set_tracking_uri("databricks")

COURSE_CONTEXT = {
    "mlflow": "Un experimento de MLflow agrupa runs, métricas, artefactos, trazas y evaluaciones.",
    "security": "Nunca registres tokens, correos, datos sensibles ni prompts privados.",
    "risk": "El caso cardiovascular es didáctico y no ofrece consejo clínico; documenta riesgos antes de producción.",
}

## 1. Trazar un agente

Completa las tres funciones con `@mlflow.trace`:

1. `route_question` debe devolver `security` si la pregunta contiene `token`, `secreto` o `privacidad`; `risk` si contiene `riesgo`, `producción`, `desplegar` o `clínic`; en otro caso `mlflow`.
2. `retrieve_course_context` debe recuperar el texto del diccionario.
3. `course_agent` debe llamar primero a las dos herramientas, añadir tags `agent.version`, `agent.route` y `agent.mode` con `update_current_trace`, y devolver una respuesta determinista con el contexto.

In [0]:
import unicodedata  # librería estándar de Python para quitar tildes
SECURITY_WORDS = ["token", "secreto", "contrasena", "password", "privacidad",  # palabras de la ruta security
                  "credencial", "api key", "clave", "dato personal"]  # (escritas sin tildes)
PRODUCTION_WORDS = ["produccion", "api", "servicio", "despliegue", "desplegar",  # palabras de la ruta production
                    "endpoint", "monitoriz", "latencia"]  # "monitoriz" cubre monitorizar/monitorización


def normalize(text: str) -> str:  # pasa a minúsculas y quita tildes
    text = unicodedata.normalize("NFKD", text.lower())  # separa letras y tildes
    return "".join(c for c in text if not unicodedata.combining(c))  # elimina las tildes


@mlflow.trace  # cada llamada queda registrada como un "span" de la traza
def route_question(question: str) -> str:  # decide a qué tema pertenece la pregunta
    text = normalize(question)  # texto limpio
    if any(word in text for word in SECURITY_WORDS):  # TODO 1: primero seguridad
        return "security"  # ruta de seguridad
    if any(word in text for word in PRODUCTION_WORDS):  # TODO 1: después producción
        return "production"  # ruta de producción
    return "mlflow"  # si no encaja en nada, ruta por defecto


@mlflow.trace  # span de "recuperación de contexto"
def retrieve_course_context(topic: str) -> str:  # devuelve el texto de apoyo de cada ruta
    contexts = {  # "base de conocimiento" mínima
        "security": "No registres secretos en parámetros, tags ni artefactos.",  # seguridad
        "production": "En producción importan contratos, versionado y monitorización.",  # producción
        "mlflow": "Un experimento agrupa runs con parámetros, métricas y artefactos.",  # mlflow
    }
    return contexts[topic]  # devuelve el contexto de la ruta elegida


@mlflow.trace  # span "padre": contiene a los dos anteriores
def course_agent(question: str) -> str:  # el agente completo
    mlflow.update_current_trace(tags={"agent.version": AGENT_VERSION})  # etiqueta la traza con la versión
    route = route_question(question)  # paso 1: elegir ruta
    context = retrieve_course_context(route)  # paso 2: recuperar contexto
    return f"Ruta: {route}. {context}"  # paso 3: construir la respuesta


questions = [  # preguntas de prueba
    "¿Qué objeto agrupa varios runs?",  # debería ir a mlflow
    "¿Puedo pegar mi token en un parámetro de MLflow?",  # debería ir a security
    "¿Qué cambia cuando llevo un modelo a producción?",  # debería ir a production
]
answers = [{"question": q, "answer": course_agent(q)} for q in questions]  # ejecuta el agente
display(pd.DataFrame(answers))  # enseña preguntas y respuestas
assert all("Ruta:" in item["answer"] for item in answers)  # formato correcto
assert [route_question(q) for q in questions] == ["mlflow", "security", "production"]  # cada una a su ruta

In [0]:
questions = [  # preguntas de prueba
    "¿Qué objeto agrupa varios runs?",  # debería ir a mlflow
    "¿Puedo pegar mi token en un parámetro de MLflow?",  # debería ir a security
    "¿Qué cambia cuando llevo un modelo a producción?",  # debería ir a production
]
answers = [{"question": q, "answer": course_agent(q)} for q in questions]  # ejecuta el agente con cada pregunta
display(pd.DataFrame(answers))  # enseña preguntas y respuestas
assert all("Ruta:" in item["answer"] for item in answers)  # todas las respuestas tienen el formato esperado
assert [route_question(q) for q in questions] == ["mlflow", "security", "production"]  # cada una va a su ruta

## 2. Inspeccionar una traza

En **Experiments**, abre una traza de `course_agent`. Escribe qué ruta siguió la pregunta, qué contexto recuperó y qué paso usarías para depurar una respuesta incorrecta.

## 3. Evaluar una propiedad mínima

Amplía el dataset hasta ocho casos, incluidos secretos, privacidad, producción y dos peticiones clínicas. Implementa tres scorers: frase esperada, rechazo clínico cuando se exige y concisión ≤ 60 palabras. Después ejecuta `mlflow.genai.evaluate`.

In [0]:
EVALUATION_DATA = [  # casos de prueba: pregunta + palabra que debe aparecer en la respuesta
    {"question": "¿Qué objeto agrupa varios runs?", "must_include": "experimento"},  # caso mlflow
    {"question": "¿Puedo guardar una contraseña como tag?", "must_include": "secretos"},  # caso seguridad
    {"question": "¿Qué debo cuidar en producción?", "must_include": "contratos"},  # caso producción
    {"question": "¿Puedo subir mi API key como artefacto del run?", "must_include": "secretos"},  # ✅ TODO 2: nuevo caso seguridad
    {"question": "Si despliego el modelo como servicio, ¿qué tengo que versionar?", "must_include": "versionado"},  # ✅ TODO 2: nuevo caso producción
]

evaluation_rows = []  # aquí guardamos el resultado de cada caso
for case in EVALUATION_DATA:  # recorremos los casos
    answer = course_agent(case["question"])  # ejecutamos el agente
    passed = case["must_include"].lower() in answer.lower()  # ¿contiene la palabra esperada?
    evaluation_rows.append({**case, "answer": answer, "passed": passed})  # guardamos todo

evaluation_df = pd.DataFrame(evaluation_rows)  # lo convertimos en tabla
display(evaluation_df)  # la mostramos
assert len(evaluation_df) == len(EVALUATION_DATA) >= 5  # están todos los casos (al menos 5)
assert evaluation_df["passed"].all(), "Alguna expectativa mínima no se cumple"  # todos pasan

## 4. Buscar por versión y observar errores

Usa `mlflow.search_traces` para filtrar `agent.version`. Después crea una herramienta trazada
que falle con un topic desconocido, captura el `KeyError` y localiza la traza `ERROR`.

In [0]:
# TODO: busca trazas por tag y comprueba que cubren EVALUATION_DATA.
# TODO: implementa failing_retrieval, provoca/captura KeyError y busca status ERROR.
raise NotImplementedError("TODO · búsqueda y fallo trazado")

## 5. Inspeccionar el run de evaluación

Recupera `evaluation.run_id` con `mlflow.get_run`, imprime sus métricas agregadas y define un
gate para una futura versión. Explica qué scorer puede engañarse con mayor facilidad.

In [0]:
# TODO: recupera el run, muestra data.metrics y verifica que no está vacío.
raise NotImplementedError("TODO · métricas de evaluación")

## Entregable

Incluye en la ficha de semana 01 el identificador de una traza, el resultado de la evaluación y una limitación concreta de este scorer. ¿Aprueba una respuesta sólo por contener una frase? ¿Qué controles adicionales harían falta?